<a href="https://colab.research.google.com/github/mibucko/ml-product-categories/blob/main/product_categories_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is part of a machine learning project in which we are developing a model to predict the product category based on the input data. The documents are available on GitHub at the following link:
https://github.com/mibucko/ml-product-categories

1. Raw data gathering

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/mibucko/ml-product-categories/main/products.csv"
df = pd.read_csv(url)

1.1. Column names standardization

In [2]:
df.columns = (df.columns.str.strip().str.lower().str.replace(r"[\s_]+", "_", regex=True).str.strip("_"))
print(df.columns)

Index(['product_id', 'product_title', 'merchant_id', 'category_label',
       'product_code', 'number_of_views', 'merchant_rating', 'listing_date'],
      dtype='object')


1.2. Brief view on data

In [3]:
print("Number of rows:", len(df))
print("Sample rows:")
print(df.sample(5))

Number of rows: 35311
Sample rows:
       product_id                                      product_title  \
17357       27245  indesit dfg15b1s 13 place freestanding dishwas...   
1010         1011  apple mrta2b/a 5.5 iphone 8 plus 256gb product...   
35004       47032  hotpoint ctf55p 55cm undercounter fridge with ...   
32549       44355                            liebherr ik2764 fridges   
24049       34765                          blomberg fnt9673p freezer   

       merchant_id category_label product_code  number_of_views  \
17357           15    Dishwashers   IT-5927-IX           2369.0   
1010            23  Mobile Phones   NO-8517-HR           4858.0   
35004          128        Fridges   CO-8231-DH           4398.0   
32549          293        Fridges   MB-4493-KY             17.0   
24049          293       Freezers   MY-5055-VM           2070.0   

       merchant_rating listing_date  
17357              1.9    6/22/2023  
1010               1.9     9/2/2023  
35004          

1.3. Dropping columns that definitely cannot influence product category: product_id, merchant_id, number_of_views, merchant_rating and listing_date.

In [4]:
df = df.drop(columns=["product_id", "merchant_id", "number_of_views",
    "merchant_rating", "listing_date"])

2. Exploratory data analysis – EDA

2.1. What categories are present in dataset?

In [5]:
print("Number of categories:", df["category_label"].nunique())
print(df["category_label"].value_counts())

Number of categories: 13
category_label
Fridge Freezers     5495
Washing Machines    4036
Mobile Phones       4020
CPUs                3771
TVs                 3564
Fridges             3457
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
fridge               123
CPU                   84
Mobile Phone          55
Name: count, dtype: int64


2.2. Is the product code related to the product category?

We choose two categories as an example. By visual inspection, we can see that there is no apparent correlation between the product code and the product category. Therefore, we also drop this column from the dataset.

In [6]:
mobile_phones = df[df["category_label"] == "Mobile Phones"]
cameras = df[df["category_label"] == "Digital Cameras"]
print('mobile phones')
print(mobile_phones[["product_code", "category_label"]].sample(20))
print('cameras')
print(cameras[["product_code", "category_label"]].sample(20))

mobile phones
     product_code category_label
111    IP-6428-FK  Mobile Phones
2723   VT-0641-QH  Mobile Phones
3757   UA-3721-CS  Mobile Phones
4034   ZR-7706-UJ  Mobile Phones
93     HY-8543-IZ  Mobile Phones
4064   PV-6911-FO  Mobile Phones
491    XN-3844-PN  Mobile Phones
3963   UZ-8533-TF  Mobile Phones
2467   RV-5014-TS  Mobile Phones
2049   JZ-5955-KB  Mobile Phones
1718   XX-2043-QL  Mobile Phones
269    BA-1447-OW  Mobile Phones
1576   YP-1899-OX  Mobile Phones
1146   DL-9029-KO  Mobile Phones
1584   YS-5429-XC  Mobile Phones
3221   ZN-2937-AL  Mobile Phones
727    NQ-2951-JL  Mobile Phones
1141   RG-8093-TE  Mobile Phones
1040   NF-0821-HV  Mobile Phones
3172   KY-4198-AC  Mobile Phones
cameras
      product_code   category_label
12367   VQ-5520-MY  Digital Cameras
13192   MV-2575-FU  Digital Cameras
11816   GO-9011-BR  Digital Cameras
13924   RP-2145-LD  Digital Cameras
13150   PW-7051-ZU  Digital Cameras
13573   MS-8382-RX  Digital Cameras
12190   XH-2355-LE  Digital Camer

In [7]:
df = df.drop(columns=["product_code"])
print('Remaining columns:')
print(df.columns)

Remaining columns:
Index(['product_title', 'category_label'], dtype='object')


3. Data processing

3.1 Merging similar categories

In [8]:
df["category_label"] = df["category_label"].replace({
    "Mobile Phone": "Mobile Phones",
    "CPU": "CPUs",
    "fridge": "Fridges"
})
print(df["category_label"].value_counts())

category_label
Fridge Freezers     5495
Mobile Phones       4075
Washing Machines    4036
CPUs                3855
Fridges             3580
TVs                 3564
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
Name: count, dtype: int64


3.2. Handling Missing Values - isna

There are 171 rows without a product title, so we drop these rows. There are also 43 rows without a product category. We remove these rows from the original dataset and store them separately in a dataset called "only_title", so that we can potentially use them later to test our model.

In [9]:
print("product_title:", df["product_title"].isna().sum())
print("category_label:", df["category_label"].isna().sum())
only_title = df[df["category_label"].isna()]

product_title: 172
category_label: 44


3.2. Handling Missing Values - dropna

We drop rows with missing values from the original dataset.

In [10]:
df = df.dropna(subset=["product_title", "category_label"])

4. Feature engineering

4.1 Brief look at product titles

In [11]:
sample = df.sample(10).sort_values("category_label")
print(sample[["category_label", "product_title"]].to_string(index=False))

category_label                                                                     product_title
          CPUs               hewlett packard enterprise intel xeon e7330 2.4ghz 6mb l2 processor
          CPUs amd cpu am2 7750 athlon 64 x2 dual core oem boxed inc fan heatsink socket am2 am2
          CPUs     intel core i7 6770hq processor 6m cache up to 3.50 ghz 2.6ghz 6mb smart cache
          CPUs                     hewlett packard enterprise dl380p gen8 intel xeon e5 2665 kit
          CPUs                  intel core i9 i9 7940x tetradeca core 14 core 3.10 ghz processor
    Microwaves                                          akai a24001b manual microwave black 800w
    Microwaves                                         bosch hmt75m654b steel built in microwave
    Microwaves                                            bosch hmt84m451 arbeitsfl che 25l 900w
           TVs                                            samsung qe49q6fn 49 qled 4k television
           TVs                

4.2. Creating new Columns

We create five new features from the product title:

- word_count – number of words in the product title
- char_count – number of characters in the product title
- special_count – number of special characters in the product title
- words_with_digits – number of words containing digits in the title
- max_word_length – length of the longest word in the title

In [12]:
df["word_count"] = df["product_title"].str.split().str.len()
df["char_count"] = df["product_title"].str.replace(" ", "").str.len()
df["special_count"] = df["product_title"].str.count(r"[^A-Za-z0-9\s]")
df["words_with_digits"] = df["product_title"].str.split().apply(
    lambda words: sum(any(char.isdigit() for char in word) for word in words))
df["max_word_length"] = df["product_title"].str.split().apply(
    lambda words: max(len(word) for word in words))

5. Data Preparation for Algorithm Training

5.1 Firstly we define seven experimental data sets:

In [13]:
y = df["category_label"]

feature_sets = {
    "baseline": ["product_title"],
    "word_count": ["product_title", "word_count"],
    "char_count": ["product_title", "char_count"],
    "special_count": ["product_title", "special_count"],
    "words_with_digits": ["product_title", "words_with_digits"],
    "max_word_length": ["product_title", "max_word_length"],
    "all_features": [
        "product_title",
        "word_count",
        "char_count",
        "special_count",
        "words_with_digits",
        "max_word_length"
    ]
}

5.2 Train-Test Split

In [14]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

5.3. TF-IDF Vectorization: We choose the unigram + bigram option

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True
)

X_train_text = tfidf.fit_transform(
    df.loc[train_idx, "product_title"]
)

X_test_text = tfidf.transform(
    df.loc[test_idx, "product_title"]
)

5.4 Scaling Numerical Features

In [16]:
from sklearn.preprocessing import MinMaxScaler

numeric_features = [
    "word_count",
    "char_count",
    "special_count",
    "words_with_digits",
    "max_word_length"
]

scaler = MinMaxScaler()

X_train_num = scaler.fit_transform(
    df.loc[train_idx, numeric_features]
)

X_test_num = scaler.transform(
    df.loc[test_idx, numeric_features]
)

5.5. Defining a Function for Feature Selection

We will call this function each time we prepare the features for algorithm training.

In [17]:
def get_feature_sets(features):

    if len(features) == 0:
        X_train_final = X_train_text
        X_test_final = X_test_text

    else:
        indices = [numeric_features.index(f) for f in features]

        X_train_final = hstack([
            X_train_text,
            X_train_num[:, indices]
        ])

        X_test_final = hstack([
            X_test_text,
            X_test_num[:, indices]
        ])

    return X_train_final, X_test_final

5.6. Experimental Feature Sets

In [20]:
feature_sets = {
    "Baseline": [],
    "Word count": ["word_count"],
    "Char count": ["char_count"],
    "Special count": ["special_count"],
    "Words with digits": ["words_with_digits"],
    "Max word length": ["max_word_length"],
    "All features": numeric_features
}

5.7. Defining a Function for Training and Evaluation

We will call this function each time we train and evaluate a new model.

In [18]:
def train_and_evaluate(model, model_name):

    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)

    print(f"\n{'='*60}")
    print(f"{model_name} - {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=model.classes_,
        columns=model.classes_
    ))

6. Training and Testing of the Algorithms

We have seven experimental datasets. We will perform training and testing using five different algorithms. Each Colab cell is dedicated to one algorithm and includes all seven experimental datasets.

6.1 Logistic Regression

Please find the metric results below. To select the optimal dataset, we focus our attention on the category with the lowest F1-score. Across all feature sets, the lowest score belongs to the **Fridges** category. The **Word count** dataset slightly outperforms the others (F1-score = 0.91), which is why we propose using it for further experimentation with this algorithm.

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from scipy.sparse import hstack
from sklearn.metrics import confusion_matrix

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = LogisticRegression(max_iter=1000)
    train_and_evaluate(model, "Logistic Regression")


Logistic Regression - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      1.00       766
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.93      0.96      0.95       681
        Freezers       0.99      0.93      0.96       440
 Fridge Freezers       0.96      0.94      0.95      1094
         Fridges       0.89      0.92      0.90       712
      Microwaves       0.99      0.95      0.97       466
   Mobile Phones       0.97      0.99      0.98       812
             TVs       0.97      0.99      0.98       708
Washing Machines       0.95      0.94      0.94       803

        accuracy                           0.96      7020
       macro avg       0.96      0.96      0.96      7020
    weighted avg       0.96      0.96      0.96      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               761                0            0         0   
Dig

6.2 Naive Bayes

Please find the metric results below. To select the optimal dataset, we focus our attention on the category with the lowest F1-score. Across all feature sets, the lowest score belongs to the Freezers category. The Baseline dataset and dataset where special characters were counted, slightly outperform the others (F1-score = 0.80), which is why we propose using these two datasets for further experimentation with this algorithm.

In [23]:
from sklearn.naive_bayes import MultinomialNB

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = MultinomialNB()
    train_and_evaluate(model, "Naive Bayes")


Naive Bayes - Baseline
                  precision    recall  f1-score   support

            CPUs       0.99      1.00      1.00       766
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.99      0.94      0.96       681
        Freezers       1.00      0.67      0.80       440
 Fridge Freezers       0.80      0.99      0.88      1094
         Fridges       0.94      0.87      0.90       712
      Microwaves       0.99      0.96      0.98       466
   Mobile Phones       0.99      0.99      0.99       812
             TVs       0.98      0.99      0.99       708
Washing Machines       0.98      0.96      0.97       803

        accuracy                           0.95      7020
       macro avg       0.97      0.94      0.95      7020
    weighted avg       0.96      0.95      0.95      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               765                0            0         0   
Digital Cam

6.3. Decision Tree

In [24]:
from sklearn.tree import DecisionTreeClassifier

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = DecisionTreeClassifier()
    train_and_evaluate(model, "Decision Tree")


Decision Tree - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      1.00       766
 Digital Cameras       0.99      0.94      0.97       538
     Dishwashers       0.94      0.90      0.92       681
        Freezers       0.89      0.91      0.90       440
 Fridge Freezers       0.93      0.89      0.91      1094
         Fridges       0.86      0.87      0.86       712
      Microwaves       0.88      0.93      0.90       466
   Mobile Phones       0.93      0.98      0.95       812
             TVs       0.93      0.96      0.94       708
Washing Machines       0.93      0.92      0.93       803

        accuracy                           0.93      7020
       macro avg       0.93      0.93      0.93      7020
    weighted avg       0.93      0.93      0.93      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               761                0            0         0   
Digital C